# 03 — Dataset Limpio (RT_IOT2022)

**Responsable:** Edwin  
**Objetivo:** Construir el primer dataset limpio reproducible aplicando las decisiones de preprocesamiento acordadas por el equipo.  
**Decisiones de referencia:** `reports/decisiones_preparacion.md`  
**Fuente de `REDUNDANT_COLS_DROP`:** `notebooks/03_matriz_correlacion.ipynb` celda `c8`

---

### Pasos
1. Cargar dataset original (84 columnas)
2. Eliminar columna constante (`bwd_URG_flag_count`)
3. Aplicar `REDUNDANT_COLS_DROP` (46 columnas)
4. Verificar conteo final de columnas (esperado: 37)
5. Eliminar filas duplicadas
6. Verificar ausencia de valores faltantes
7. Verificar y corregir tipos de datos
8. Resumen y respuestas
9. Guardar `data/processed/dataset_limpio.csv`

In [ ]:
import zipfile
import os
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

ZIP = '../data/rt-iot2022.zip'
OUT = '../data/processed/dataset_limpio.csv'
os.makedirs('../data/processed', exist_ok=True)

print('Setup completo.')

## 1. Carga del dataset original

In [ ]:
with zipfile.ZipFile(ZIP) as z:
    with z.open('RT_IOT2022') as f:
        df = pd.read_csv(f, index_col=0)

print(f'Dataset original: {df.shape[0]:,} filas \u00d7 {df.shape[1]} columnas')
df.head(3)

## 2. Eliminar columna constante

`bwd_URG_flag_count` es 100% ceros (varianza cero). Confirmado en `eda_edwin_calidad_datos.ipynb` sección 8 y en `03_matriz_correlacion.ipynb` celda `c1`. Decisión E1 de `reports/decisiones_preparacion.md`.

In [ ]:
CONSTANT_DROP = ['bwd_URG_flag_count']

# Verificar que siguen siendo constantes
const_confirmadas = [c for c in CONSTANT_DROP if df[c].nunique() <= 1]
print(f'Columnas constantes confirmadas: {const_confirmadas}')

df = df.drop(columns=CONSTANT_DROP)
print(f'Tras eliminar constantes: {df.shape[1]} columnas  (esperado: 83)')

## 3. Aplicar `REDUNDANT_COLS_DROP` (lista oficial — 46 columnas)

Lista acordada por el equipo en `notebooks/03_matriz_correlacion.ipynb` celda `c8`.  
Método: clustering jerárquico Spearman + complete linkage, corte |ρ|=0.9. Regla del representante: no-leaky → más denso → más interpretable. Ver `reports/decisiones_preparacion.md` sección F.

In [ ]:
REDUNDANT_COLS_DROP = [
    # G1 (reps conservados = fwd_pkts_tot + bwd_pkts_payload.avg)
    'bwd_data_pkts_tot', 'bwd_pkts_payload.max', 'bwd_pkts_payload.tot', 'bwd_pkts_payload.std',
    'fwd_iat.min', 'fwd_iat.max', 'fwd_iat.tot', 'fwd_iat.avg',
    'bwd_iat.min', 'bwd_iat.max', 'bwd_iat.tot', 'bwd_iat.avg',
    'flow_iat.std', 'fwd_subflow_pkts', 'bwd_subflow_bytes',
    # G2 (rep conservado = flow_duration)
    'flow_iat.min', 'flow_iat.max', 'flow_iat.tot', 'flow_iat.avg',
    'active.min', 'active.max', 'active.tot', 'active.avg',
    # G3 (reps conservados = bwd_init_window_size + fwd_iat.std)
    'bwd_PSH_flag_count', 'fwd_pkts_payload.std', 'bwd_iat.std',
    # G4 (rep conservado = flow_pkts_per_sec)
    'fwd_pkts_per_sec', 'bwd_pkts_per_sec', 'payload_bytes_per_second',
    # G5 (reps conservados = fwd_header_size_max + fwd_init_window_size)
    'fwd_header_size_tot', 'fwd_header_size_min',
    # G6 (reps conservados = bwd_header_size_max + flow_ACK_flag_count)
    'bwd_header_size_tot', 'bwd_header_size_min',
    # G7 idle (rep conservado = idle.tot)
    'idle.min', 'idle.max', 'idle.avg',
    # G8 (rep conservado = fwd_pkts_payload.avg)
    'fwd_pkts_payload.min',
    # G9 (rep conservado = fwd_subflow_bytes)
    'fwd_pkts_payload.max', 'fwd_pkts_payload.tot',
    # G10 fwd_bulk (rep conservado = fwd_bulk_bytes)
    'fwd_bulk_packets', 'fwd_bulk_rate',
    # G11 bwd_bulk — FAMILIA ENTERA eliminada (mezclada, sin señal clara — decisión G1)
    'bwd_bulk_bytes', 'bwd_bulk_packets', 'bwd_bulk_rate',
    # G12 (rep conservado = bwd_pkts_tot)
    'bwd_subflow_pkts',
    # G13 (rep conservado = flow_CWR_flag_count)
    'flow_ECE_flag_count',
]

# Verificar que todas existen antes de eliminar
faltan = [c for c in REDUNDANT_COLS_DROP if c not in df.columns]
assert not faltan, f'Columnas inexistentes en el dataset: {faltan}'
print(f'REDUNDANT_COLS_DROP: {len(REDUNDANT_COLS_DROP)} columnas — todas verificadas \u2714')

df = df.drop(columns=REDUNDANT_COLS_DROP)
print(f'Tras eliminar redundantes: {df.shape[1]} columnas  (esperado: 37)')

## 4. Verificar columnas finales

In [ ]:
EXPECTED_COLS = 37
assert df.shape[1] == EXPECTED_COLS, f'Esperado {EXPECTED_COLS} columnas, obtenido {df.shape[1]}'
print(f'\u2714 Columnas finales: {df.shape[1]}')
print()
print('Columnas que permanecen en el dataset limpio:')
for i, c in enumerate(df.columns, 1):
    print(f'  {i:2d}. {c}  ({df[c].dtype})')

## 5. Eliminar filas duplicadas

Decisión E2: `drop_duplicates()`. Las filas duplicadas exactas (~1.5%) pueden sesgar métricas si la misma fila queda en train y test. Ver `reports/decisiones_preparacion.md`.

In [ ]:
n_antes = len(df)
df = df.drop_duplicates()
n_dup = n_antes - len(df)

print(f'Filas antes           : {n_antes:,}')
print(f'Filas duplicadas      : {n_dup:,} ({n_dup / n_antes * 100:.2f}%)')
print(f'Filas tras limpieza   : {len(df):,}')

## 6. Verificar valores faltantes

El EDA de calidad confirmó 0 nulos. Este paso verifica que sigan siendo 0 tras la limpieza. Decisión E3.

In [ ]:
nulos = df.isnull().sum()
nulos_presentes = nulos[nulos > 0]

if nulos_presentes.empty:
    print('\u2714 Sin valores faltantes — dataset completo (0 nulos en todas las columnas)')
else:
    print(f'\u26a0 {len(nulos_presentes)} columnas con nulos:')
    display(nulos_presentes.to_frame('nulos'))

## 7. Verificar y corregir tipos de datos

- `proto`, `service`, `Attack_type` → convertir a `category` (ahorro de memoria ~60-70%)
- El resto → numéricos (`int64` / `float64`)
- Las columnas leaky (`id.orig_p`, `id.resp_p`, `service`, `proto`) se **conservan** con advertencia. Ver decisión H1 en `reports/decisiones_preparacion.md`.

In [ ]:
cat_cols = ['proto', 'service', 'Attack_type']

print('Tipos ANTES de conversión:')
print(df[cat_cols].dtypes.to_string())

for col in cat_cols:
    df[col] = df[col].astype('category')

print()
print('Tipos DESPUÉS de conversión:')
print(df[cat_cols].dtypes.to_string())

num_cols = df.select_dtypes(include='number').columns.tolist()
print(f'\nColumnas numéricas ({len(num_cols)}): tipos correctos \u2714')

# Advertencia sobre columnas leaky
LEAKY_COLS = ['id.orig_p', 'id.resp_p', 'service', 'proto']
leaky_presentes = [c for c in LEAKY_COLS if c in df.columns]
print(f'\n\u26a0  Columnas leaky conservadas (toggle — decisión H1): {leaky_presentes}')
print('   Usar include_leaky=False en build_preprocessor() al modelar para excluirlas.')

## 8. Resumen del preprocesamiento

In [ ]:
print('=' * 62)
print('RESUMEN DEL PREPROCESAMIENTO  —  RT_IOT2022')
print('=' * 62)

print(f'\n\u00bfCu\u00e1ntas columnas ten\u00eda originalmente el dataset?')
print(f'  → 84 columnas')

print(f'\n\u00bfCu\u00e1ntas columnas quedaron despu\u00e9s del preprocesamiento?')
print(f'  → {df.shape[1]} columnas')

print(f'\n\u00bfQu\u00e9 columnas fueron eliminadas?')
print(f'  → 1 constante     : bwd_URG_flag_count (100% ceros, varianza cero)')
print(f'  → 46 redundantes  : REDUNDANT_COLS_DROP (grupos G1–G13, ver doc adjunto)')
print(f'  → Total eliminadas: {84 - df.shape[1]} columnas')

print(f'\nFilas originales                  : 123,117')
print(f'Filas tras drop_duplicates        : {len(df):,}')
print(f'Filas duplicadas eliminadas       : {123117 - len(df):,}')

print(f'\nValores faltantes                 : 0 \u2714')

print(f'\n\u00bfQu\u00e9 problemas persisten despu\u00e9s de la limpieza?')
counts = df['Attack_type'].value_counts()
print(f'  → Desbalance extremo: {str(counts.index[0])} representa el {counts.iloc[0]/len(df)*100:.1f}% del dataset')
print(f'    Ratio mayor/menor clase: {counts.iloc[0] // counts.iloc[-1]:,}:1')
print(f'  → Columnas leaky presentes: {leaky_presentes}')
print(f'    (pueden inflar métricas si se usan sin el toggle — decisión H1)')

print()
print(f'Shape final del dataset limpio: {df.shape[0]:,} filas \u00d7 {df.shape[1]} columnas')

## 9. Guardar dataset limpio

In [ ]:
df.to_csv(OUT, index=True)
print(f'Guardado en   : {OUT}')
print(f'Shape guardado: {df.shape[0]:,} filas \u00d7 {df.shape[1]} columnas')

# Verificación de integridad
df_check = pd.read_csv(OUT, index_col=0)
assert df_check.shape == df.shape, f'Error en guardado: shape esperado {df.shape}, leído {df_check.shape}'
print('\u2714 Verificación de lectura exitosa — el archivo es íntegro')